# Quickstart

Use `onceover.profile(df)` to inspect an orders table for candidate keys,
relationships between columns, and duplicate rows.

````{admonition} Minimal example
:class: tip

```python
import polars as pl
import onceover

report = onceover.profile(df)      # budget="standard" by default
print(report.overview["headline"]) # one-glance summary
report.findings                    # ranked statements with evidence
report.columns                     # per-column cards
report.coverage                    # what was examined, skipped, capped
```
````

First we need a table. The frame below is small but not clean: it has a
proper primary key, a region lookup, one nearly-unique external reference
with two duplicated values, and a column derived from another. Fixtures like
this are planted with known structure so that every part of the report can
be checked against the truth.

In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(7)
n = 2000

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
order_id = list(range(n))
external_ref = [f"R{i:05}" for i in range(n)]
external_ref[-1] = external_ref[0]   # two duplicated values
external_ref[-2] = external_ref[1]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
})
df.head(4)

## Profile it

One call runs the detectors enabled by the default budget.
[](large_tables.ipynb) describes runtime and coverage on larger tables.

In [ ]:
report = onceover.profile(df)
print(report.overview["headline"])

The headline summarizes the highest-ranked findings: a key candidate
exists, five exact functional dependencies were verified (one of them with
a formula), several columns are predictable from others, and conditional
rules summarize the lookup structure. Each clause corresponds to findings
deeper in the ledger.

## Read the ledger

Findings are ranked by interestingness, most interesting first. Every
finding is one claim with its evidence attached. The top of the ledger on
this table:

In [ ]:
for f in report.findings[:8]:
    print(f"[{f['interestingness']:.2f}] {f['claim'][:95]}")

The report contains two exact relationships and one affine identity for
`quantity` and `double_quantity`, plus a region lookup. It also reports: `order_id`
uniquely identifies rows in this fixture, while `external_ref` is
nearly unique but carries two repeated values.

Ranking orders findings by the scoring policy. The ledger groups
statements into categories — dependencies, uniqueness, column roles, and so
on — and a histogram over them is the next thing to look at:

In [ ]:
from collections import Counter

print(Counter((f["category"], f.get("type")) for f in report.findings))

To inspect findings about one column, filter by `subject.columns`. For example, these findings refer to `external_ref`:

In [ ]:
for f in report.findings:
    if "external_ref" in f["subject"]["columns"]:
        print(f"[{f['category']}] {f['claim']}")

These findings describe separate properties of `external_ref`: `external_ref` looks like a unique identifier
but carries two repeated values. A join that assumes it is unique can
produce additional rows. The findings chapter returns to these field-by-field.

## One finding in full

Here is a complete finding, the way a program would consume it:

In [ ]:
f = report.findings[0]
print(json.dumps(f, indent=1)[:1600])

The finding has an `id`, a `category` and optional `type`, a `subject` naming the columns, and a `specification` defining the claim. Its `claim` is a readable summary. The `evidence` list records the method, scope, validation, and metrics. `interestingness` determines its ranking; `interpretation`, when present, suggests a possible action. See the [findings chapter](findings.ipynb) for details.

In [ ]:
identity = next(f for f in report.findings if "within tolerance" in f["claim"])
print(identity["claim"])
print("category:", identity["category"], "| type:", identity.get("type"))
print("form:", identity["specification"].get("form"))
ev = identity["evidence"][0]
print("basis:", ev["basis"], "| sampling:", ev["scope"]["sampling"])
print("validation:", ev["validation"]["method"])
print("metrics:", ev["metrics"])

That detector fit `y = a·x + b` on a sample and then checked the fitted
formula's residuals on every row: 2,000 of 2,000 matched.
The evidence chapters explain `basis` (measured, estimated, inferred),
`scope`, and `validation`: a user-facing version in
[](findings.ipynb), the full model in
[](../methodology/evidence.ipynb).

## Skim the column cards

`report.columns` holds one card per column, including its inferred role
and summary statistics. Two cards from this table:

In [ ]:
print(json.dumps(report.columns["region_id"], indent=1)[:900])
print(json.dumps(report.columns["double_quantity"], indent=1)[:700])

`region_id` is a numeric column that behaves like a category (three
distinct values), and its card says so, with the `treat_as_categorical`
action hint attached. `double_quantity`'s card carries exact summary
statistics and concentration. [](columns.ipynb) covers every field. Cards are computed over all input rows.

## Check what ran

The coverage block records what each detector examined. On this table, the default budget completes the configured passes:

In [ ]:
for name, info in report.coverage.items():
    print(f"{name:28s} {info['status']:14s} {info['detail'][:72]}")

Two entries deserve a second look. `row_duplicates` reports "no exact
duplicate rows (exact)" without scanning: since `order_id` was proven unique
and null-free, duplicates are impossible, and the profiler took the cheap
logical path. `functional_dependencies` shows its verification scale — on
larger tables this is where sampling appears, and the evidence `scope` on
each finding changes from `full` to `sample` accordingly.

## Save it

Reports serialize deterministically: the same table and configuration
produce byte-identical output when no deadline is set, so reports can be
stored and compared across runs.

In [ ]:
import os
import tempfile

path = report.save(tempfile.mkdtemp() + "/profile.json")
print("saved", os.path.getsize(path), "bytes to", path)
print(report.to_markdown()[:400])

`to_markdown()` renders the ledger, a column table, and the coverage block
as a compact document — convenient for an issue report, a pull request
description, or a message to an agent. `save()` writes JSON or Markdown to a
path; `.data` returns the whole versioned report as a dict.
[](serialization.ipynb) covers the format and its stability policy.

## Where to go next

- [](findings.ipynb) — the ledger: statements, categories, evidence,
  filtering, what to act on.
- [](columns.ipynb) — every card field, role inference, concentration.
- [](coverage.ipynb) — budgets, sampling, deadlines, detector selection.
- The
  [analysis notebook](https://github.com/fcela/onceover/blob/main/notebooks/onceover_analysis.ipynb)
  applies the same report to a larger, messier fixture with charts and
  controls.